# Variate importance: how much does each term contribute to each feature?

`0.linear_modeling.ipynb` fits, per (treatment+DMSO combo, feature), the model

```
feature ~ C(treatment) * C(patient) + cell_count + organoid_count + cell_per_organoid_count
```

and saves per-term **coefficients and p-values** (`1.EDA/results/linear_modeling/*.parquet`).
That table answers *"is term X significant, and in which direction"* but not
*"how much of this feature's variance does term X actually account for"* —
coefficients on different terms aren't on a comparable scale (a dummy
contrast vs. a per-unit slope), so they can't be ranked against each other
directly.

This notebook answers that: for every (combo, feature) model, it decomposes
the model's total sum of squares into the share attributable to each term
via a **Type II ANOVA**, and expresses each term's contribution as a
percentage of total variance explained. That is directly comparable across
terms and features, and lets us rank `treatment`, `patient`,
`treatment:patient`, `cell_count`, `organoid_count`, and
`cell_per_organoid_count` by overall importance.

Refits the same ~11k models as `0.linear_modeling.ipynb` (161 organoid +
278 single-cell features x 25 treatments), ~10 minutes total.


In [1]:
import pathlib
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from notebook_init_utils import init_notebook
from statsmodels.stats.anova import anova_lm

warnings.filterwarnings("ignore")
root_dir, in_notebook = init_notebook()

if in_notebook:
    from tqdm.notebook import tqdm
else:
    from tqdm import tqdm

results_path = pathlib.Path(root_dir, "4.linear_modeling/results/variate_importance")
results_path.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

In [2]:
profile_dict = {
    "organoid": pathlib.Path(
        root_dir,
        "data/profiles_3D/all_patients/1.feature_selected_profiles/organoid_norm_fs_profiles.parquet",
    ),
    "single_cell": pathlib.Path(
        root_dir,
        "data/profiles_3D/all_patients/1.feature_selected_profiles/sc_norm_fs_profiles.parquet",
    ),
}
organoid_counts_source = pathlib.Path(
    root_dir,
    "data/profiles_3D/all_patients/1.feature_selected_profiles/organoid_norm_fs_profiles.parquet",
)

# friendly names for the ANOVA table's term labels
TERM_RENAME = {
    "C(Metadata_treatment_full)": "treatment",
    "C(patient)": "patient",
    "C(Metadata_treatment_full):C(patient)": "treatment:patient",
    "cell_count": "cell_count",
    "organoid_count": "organoid_count",
    "cell_per_organoid_count": "cell_per_organoid_count",
    "Residual": "residual",
}

## Load + prep, exactly mirroring `0.linear_modeling.ipynb`

Same patient exclusion, same combined treatment+dose label, same count
covariates, same texture-feature drop, same outlier clipping — so these
variance-partitioning results line up 1:1 with the existing coefficient
table.


In [3]:
def load_and_prep(profile_name):
    df = pd.read_parquet(profile_dict[profile_name])
    df = df.rename(
        columns={
            "Metadata_Biology_PatientTumor": "patient",
            "Metadata_Experiment_Treatment": "treatment",
        }
    )
    df = df.loc[df["patient"] != "NF0037_T1_CQ1"]
    df["Metadata_treatment_full"] = (
        df["treatment"].astype(str)
        + "_"
        + df["Metadata_Experiment_Dose"].astype(str)
        + df["Metadata_Experiment_Unit"].astype(str)
    )
    treatment_meta = (
        df[
            [
                "Metadata_treatment_full",
                "treatment",
                "Metadata_Experiment_TherapeuticCategories",
            ]
        ]
        .drop_duplicates()
        .set_index("Metadata_treatment_full")
    )

    if profile_name == "single_cell":
        organoid_counts_df = pd.read_parquet(
            organoid_counts_source,
            columns=[
                "Metadata_Biology_PatientTumor",
                "Metadata_Experiment_Well",
                "Metadata_WellOrganoidCount",
            ],
        ).drop_duplicates()
        organoid_counts_df = organoid_counts_df.rename(
            columns={"Metadata_Biology_PatientTumor": "patient"}
        )
        df = df.merge(
            organoid_counts_df, on=["patient", "Metadata_Experiment_Well"], how="left"
        )
        df = df.dropna(subset=["Metadata_WellOrganoidCount"])
        df["cell_count"] = df["Metadata_Object_WellSingleCellCount"]
        df["organoid_count"] = df["Metadata_WellOrganoidCount"]
    else:
        df["cell_count"] = df["Metadata_Object_OrganoidSingleCellCount"]
        df["organoid_count"] = df["Metadata_WellOrganoidCount"]
    df["cell_per_organoid_count"] = df["cell_count"] / df["organoid_count"]

    count_columns = ["cell_count", "organoid_count", "cell_per_organoid_count"]
    metadata_columns = (
        ["patient", "treatment"]
        + count_columns
        + [c for c in df.columns if c.startswith("Metadata_")]
    )
    df = df.drop(columns=[c for c in df.columns if "_Texture_" in c])
    feature_columns = [c for c in df.columns if c not in metadata_columns]
    df[feature_columns] = df[feature_columns].clip(lower=-1e1, upper=1e1)
    for col in df.columns:
        new_col = col.replace(".", "")
        df.rename(columns={col: new_col}, inplace=True)
    feature_columns = [c.replace(".", "") for c in feature_columns]

    dmso_label = df.loc[df["treatment"] == "DMSO", "Metadata_treatment_full"].unique()[
        0
    ]
    combo_list = [
        (dmso_label, t)
        for t in df["Metadata_treatment_full"].unique()
        if t != dmso_label
    ]
    return df, feature_columns, treatment_meta, combo_list

In [4]:
def variance_partition(df, feature_columns, treatment_meta, combo_list, profile_name):
    rows = {
        "profile": [],
        "treatment": [],
        "drug": [],
        "therapeutic_category": [],
        "feature": [],
        "term": [],
        "sum_sq": [],
        "df": [],
        "fvalue": [],
        "pvalue": [],
        "pct_variance_explained": [],
        "rsquared": [],
    }
    for combo in tqdm(
        combo_list, desc=f"{profile_name}: treatment combos", unit="combo"
    ):
        drug_name = treatment_meta.loc[combo[1], "treatment"]
        therapeutic_category = treatment_meta.loc[
            combo[1], "Metadata_Experiment_TherapeuticCategories"
        ]

        df_trt = df.loc[df["Metadata_treatment_full"].isin(combo)].copy()
        df_trt["Metadata_treatment_full"] = pd.Categorical(
            df_trt["Metadata_treatment_full"], categories=list(combo)
        )
        patients_in_combo = sorted(df_trt["patient"].unique())
        df_trt["patient"] = pd.Categorical(
            df_trt["patient"], categories=patients_in_combo
        )

        for col in tqdm(feature_columns, desc="features", unit="feature", leave=False):
            formula = (
                f"Q('{col}') ~ C(Metadata_treatment_full) * C(patient)"
                " + cell_count + organoid_count + cell_per_organoid_count"
            )
            model = smf.ols(formula=formula, data=df_trt)
            results = model.fit()
            aov = anova_lm(results, typ=2)
            total_ss = aov["sum_sq"].sum()

            for term_label, term_row in aov.iterrows():
                term = TERM_RENAME.get(term_label, term_label)
                rows["profile"].append(profile_name)
                rows["treatment"].append(combo[1])
                rows["drug"].append(drug_name)
                rows["therapeutic_category"].append(therapeutic_category)
                rows["feature"].append(col)
                rows["term"].append(term)
                rows["sum_sq"].append(term_row["sum_sq"])
                rows["df"].append(term_row["df"])
                rows["fvalue"].append(term_row.get("F", np.nan))
                rows["pvalue"].append(term_row.get("PR(>F)", np.nan))
                rows["pct_variance_explained"].append(
                    term_row["sum_sq"] / total_ss * 100
                )
                rows["rsquared"].append(results.rsquared)
    return pd.DataFrame(rows)

In [5]:
def split_feature_name(pdf):
    pdf = pdf.copy()
    pdf[["Compartment", "Channel", "Feature_type", "Measurement"]] = pdf[
        "feature"
    ].str.split("_", n=3, expand=True)
    pdf.loc[pdf["Feature_type"] == "AreaSizeShape", "Measurement"] = pdf["Channel"]
    pdf.loc[pdf["Feature_type"] == "AreaSizeShape", "Channel"] = None
    return pdf


partition_dfs = {}
for profile_name in profile_dict:
    out_path = results_path / f"{profile_name}_variance_partition.parquet"
    if out_path.exists():
        print(f"{profile_name}: loading cached fit from {out_path}")
        pdf = pd.read_parquet(out_path)
    else:
        df, feature_columns, treatment_meta, combo_list = load_and_prep(profile_name)
        pdf = variance_partition(
            df, feature_columns, treatment_meta, combo_list, profile_name
        )
        pdf.to_parquet(out_path, index=False)
        print(f"{profile_name}: {pdf.shape} -> {out_path}")
    partition_dfs[profile_name] = split_feature_name(pdf)

variance_df = pd.concat(partition_dfs.values(), ignore_index=True)
print(variance_df.head())

organoid: treatment combos:   0%|          | 0/25 [00:00<?, ?combo/s]

features:   0%|          | 0/161 [00:00<?, ?feature/s]

features:   0%|          | 0/161 [00:00<?, ?feature/s]

features:   0%|          | 0/161 [00:00<?, ?feature/s]

features:   0%|          | 0/161 [00:00<?, ?feature/s]

...

features:   0%|          | 0/278 [00:00<?, ?feature/s]

features:   0%|          | 0/278 [00:00<?, ?feature/s]

features:   0%|          | 0/278 [00:00<?, ?feature/s]

features:   0%|          | 0/278 [00:00<?, ?feature/s]

single_cell: (48650, 12) -> /home/lippincm/Documents/fork2_NF1_organoid_profile_analysis/4.linear_modeling/results/variate_importance/single_cell_variance_partition.parquet
    profile       treatment        drug therapeutic_category  \
0  organoid  Trametinib_1uM  Trametinib     Kinase Inhibitor   
1  organoid  Trametinib_1uM  Trametinib     Kinase Inhibitor   
2  organoid  Trametinib_1uM  Trametinib     Kinase Inhibitor   
3  organoid  Trametinib_1uM  Trametinib     Kinase Inhibitor   
4  organoid  Trametinib_1uM  Trametinib     Kinase Inhibitor   

                      feature               term      sum_sq    df     fvalue  \
0  Organoid_AGP_Granularity_1          treatment    0.122627   1.0   0.034267   
1  Organoid_AGP_Granularity_1            patient  526.402967  11.0  13.372432   
2  Organoid_AGP_Granularity_1  treatment:patient  282.329369  11.0   7.172130   
3  Organoid_AGP_Granularity_1         cell_count   83.870346   1.0  23.436524   
4  Organoid_AGP_Granularity_1     org

## Which variates matter most, overall?

Average each term's share of variance explained across every (combo,
feature) model fit. This is the headline ranking: on average, how much of a
feature's variance does each term of the model account for?


In [6]:
TERM_ORDER = [
    "treatment",
    "patient",
    "treatment:patient",
    "cell_count",
    "organoid_count",
    "cell_per_organoid_count",
    "residual",
]

term_summary_dfs = {}
for name, pdf in partition_dfs.items():
    overall = (
        pdf.groupby("term")["pct_variance_explained"]
        .agg(["mean", "median", "std"])
        .reindex(TERM_ORDER)
    )
    term_summary_dfs[name] = overall
    out_path = results_path / f"{name}_term_summary.parquet"
    overall.reset_index().rename(columns={"index": "term"}).to_parquet(
        out_path, index=False
    )
    print(
        f"\n==== {name}: mean % variance explained per term (across {pdf['feature'].nunique()} features x {pdf['treatment'].nunique()} treatments) ===="
    )
    print(overall)
    print(f"saved -> {out_path}")


==== organoid: mean % variance explained per term (across 161 features x 25 treatments) ====
                              mean     median       std
term                                                   
treatment                 0.249362   0.096763  0.412724
patient                   7.808874   4.608585  8.245082
treatment:patient         1.470313   1.110826  1.379582
cell_count                2.896825   0.906045  6.376597
organoid_count            0.410830   0.114484  0.787271
cell_per_organoid_count   0.098328   0.050731  0.136579
residual                 87.115341  90.551536  9.254656
saved -> /home/lippincm/Documents/fork2_NF1_organoid_profile_analysis/4.linear_modeling/results/variate_importance/organoid_term_summary.parquet

==== single_cell: mean % variance explained per term (across 278 features x 25 treatments) ====
                              mean     median       std
term                                                   
treatment                 0.210675   0.065537  0

## Per-feature breakdown: how does each variate contribute to *every* feature?

Average each term's % variance explained across all 24 treatment combos,
per feature — a `feature x term` importance matrix. Saved in full; shown
here for the features where the model explains the most total variance
(highest R^2), since a low-R^2 feature's term breakdown is mostly noise.


In [7]:
def feature_term_matrix(pdf):
    matrix = pdf.pivot_table(
        index="feature", columns="term", values="pct_variance_explained", aggfunc="mean"
    )
    matrix = matrix.reindex(columns=TERM_ORDER)
    mean_rsq = pdf.groupby("feature")["rsquared"].mean()
    matrix["mean_rsquared"] = mean_rsq
    return matrix.sort_values("mean_rsquared", ascending=False)


feature_matrices = {}
for name, pdf in partition_dfs.items():
    matrix = feature_term_matrix(pdf)
    feature_matrices[name] = matrix
    out_path = results_path / f"{name}_feature_by_term_importance.parquet"
    matrix.reset_index().to_parquet(out_path, index=False)
    print(
        f"{name}: saved {matrix.shape[0]} features x {matrix.shape[1]} columns -> {out_path}"
    )
    print(matrix.head(15))

organoid: saved 161 features x 8 columns -> /home/lippincm/Documents/fork2_NF1_organoid_profile_analysis/4.linear_modeling/results/variate_importance/organoid_feature_by_term_importance.parquet
term                                                treatment    patient  \
feature                                                                    
Organoid_DNA_Intensity_IntegratedIntensity           0.107897   2.281414   
Organoid_DNA_Intensity_IntegratedIntensityEdge       0.127466   2.986215   
Organoid_DNA_Granularity_6                           0.100793  32.467286   
Organoid_NoChannel_AreaSizeShape_Volume              0.174862   2.260780   
Organoid_AGP_Intensity_IntegratedIntensity           0.184981   1.827186   
Organoid_ER_Intensity_IntegratedIntensity            0.173964   3.185168   
Organoid_Mito_Intensity_IntegratedIntensityEdge      0.213003   3.024804   
Organoid_ER_Intensity_IntegratedIntensityEdge        0.241376   5.680738   
Organoid_DNA-AGP_Colocalization_MandersCoeffCo

## Which channel / feature type is most driven by treatment vs. by nuisance covariates?

Roll the per-(combo, feature) contributions up to `Channel` x `Feature_type`
and compare the average share attributed to `treatment` (+ its patient
interaction) against the share attributed to the three count covariates.
Feature types dominated by counts are the ones a density-normalization
would matter most for.


In [8]:
def channel_feature_term_summary(pdf):
    pdf = pdf.copy()
    pdf["term_group"] = pdf["term"].map(
        {
            "treatment": "treatment (direct + response)",
            "treatment:patient": "treatment (direct + response)",
            "patient": "patient baseline",
            "cell_count": "count covariates",
            "organoid_count": "count covariates",
            "cell_per_organoid_count": "count covariates",
            "residual": "residual",
        }
    )
    return (
        pdf.groupby(["Channel", "Feature_type", "term_group"], dropna=False)[
            "pct_variance_explained"
        ]
        .mean()
        .reset_index()
        .pivot_table(
            index=["Channel", "Feature_type"],
            columns="term_group",
            values="pct_variance_explained",
        )
    )


channel_feature_summaries = {}
for name, pdf in partition_dfs.items():
    summary = channel_feature_term_summary(pdf)
    channel_feature_summaries[name] = summary
    out_path = results_path / f"{name}_channel_feature_term_summary.parquet"
    summary.reset_index().to_parquet(out_path, index=False)
    print(f"\n==== {name}: variance share by channel/feature-type ====")
    print(
        summary.sort_values("treatment (direct + response)", ascending=False).head(15)
    )
    print(f"saved -> {out_path}")


==== organoid: variance share by channel/feature-type ====
term_group               count covariates  patient baseline   residual  \
Channel  Feature_type                                                    
AGP      Intensity               1.498709          7.171823  85.944715   
ER       Intensity               1.694889          8.821698  83.904209   
Mito     Granularity             0.408609          5.602124  91.355953   
ER-Mito  Colocalization          0.801606         10.308966  85.480167   
Mito     Intensity               1.525497          3.171126  90.576778   
ER-AGP   Colocalization          0.937917         12.109422  83.427250   
ER       Granularity             0.296773          8.045387  89.447535   
AGP      Granularity             0.407048          4.395207  92.799171   
Mito-AGP Colocalization          0.815166         11.993973  83.992111   
DNA      Intensity               2.121866          3.578938  88.487728   
DNA-ER   Colocalization          0.673081         13

## Organoid vs. single-cell: does variate importance agree across scale?


In [9]:
compare = pd.concat(
    [
        pdf.groupby("term")["pct_variance_explained"].mean().rename(name)
        for name, pdf in partition_dfs.items()
    ],
    axis=1,
).reindex(TERM_ORDER)
compare["difference (organoid - single_cell)"] = (
    compare["organoid"] - compare["single_cell"]
)
out_path = results_path / "term_importance_organoid_vs_sc.parquet"
compare.reset_index().rename(columns={"index": "term"}).to_parquet(
    out_path, index=False
)
print(compare)
print(f"saved -> {out_path}")

                          organoid  single_cell  \
term                                              
treatment                 0.249362     0.210675   
patient                   7.808874     7.396491   
treatment:patient         1.470313     1.406377   
cell_count                2.896825     0.400673   
organoid_count            0.410830     0.286476   
cell_per_organoid_count   0.098328     0.267045   
residual                 87.115341    90.074397   

                         difference (organoid - single_cell)  
term                                                          
treatment                                           0.038686  
patient                                             0.412383  
treatment:patient                                   0.063936  
cell_count                                          2.496152  
organoid_count                                      0.124353  
cell_per_organoid_count                            -0.168717  
residual                            

## Summary

- **Ranking terms overall**: see the term-summary table above (also saved
  to `{organoid,single_cell}_term_summary.parquet`) — plotted in
  `plot_variate_importance_term_summary.ipynb`.
- **Per-feature answer**: `4.linear_modeling/results/variate_importance/
  {organoid,single_cell}_feature_by_term_importance.parquet` has, for every
  feature, the average % of its variance attributable to `treatment`,
  `patient`, `treatment:patient`, `cell_count`, `organoid_count`, and
  `cell_per_organoid_count` — plotted in
  `plot_variate_importance_feature_heatmap.ipynb`.
- **Channel/feature-type roll-up** (saved to
  `{organoid,single_cell}_channel_feature_term_summary.parquet`) flags which
  measurement families are mostly density-driven vs. treatment-driven —
  pairs with the density confound finding in `explore_leads.ipynb`.
- **Organoid vs. single-cell comparison** saved to
  `term_importance_organoid_vs_sc.parquet`, plotted in
  `plot_variate_importance_organoid_vs_sc.ipynb`.
- A large `treatment:patient` share relative to `treatment` on its own is
  itself a finding — it means response heterogeneity across patients is a
  bigger driver of that feature than the average drug effect, reinforcing
  the patient-specific-responder leads from `explore_leads.ipynb`.
